# 05 — Análise de Categorias e Preços

Este notebook explora distribuição de preços por categoria, heatmap desconto × rating por subcategoria, correlações e identifica top produtos por categoria.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR.name != 'amazon-product-intelligence' and PROJECT_DIR.parent != PROJECT_DIR:
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
REPORTS_DIR = PROJECT_DIR / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')


C:\Users\flavi\Anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Carregar base de produtos com clusters

In [2]:
df = pd.read_csv(PROCESSED_DIR / 'base_produtos_com_clusters.csv')
df.shape


(1351, 22)

## Boxplots de preço por categoria

In [3]:
top_cats = df['main_category'].value_counts().head(10).index
d = df[df['main_category'].isin(top_cats)].dropna(subset=['discounted_price_clean'])
plt.figure(figsize=(14, 6))
sns.boxplot(data=d, x='main_category', y='discounted_price_clean')
plt.xticks(rotation=45, ha='right')
plt.title('Discounted Price by Category (Top 10)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'boxplot_price_by_category.png', dpi=160)
plt.close()


## Heatmap desconto × rating por subcategoria

In [4]:
sub = df.dropna(subset=['sub_category','discount_pct_clean','rating_clean']).copy()
sub = sub[sub['main_category'].isin(top_cats)]
pivot = (
    sub.groupby(['sub_category'], as_index=False)[['discount_pct_clean','rating_clean']]
    .mean(numeric_only=True)
    .set_index('sub_category')
    .sort_values('discount_pct_clean', ascending=False)
)
plt.figure(figsize=(10, max(6, 0.25 * len(pivot))))
sns.heatmap(pivot, annot=False, cmap='mako')
plt.title('Subcategory: Avg Discount vs Avg Rating')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'heatmap_discount_rating_by_subcategory.png', dpi=160)
plt.close()


## Correlações e Top produtos por categoria

In [5]:
num_cols = ['discounted_price_clean','actual_price_clean','discount_pct_clean','rating_clean','rating_count_clean','economia_absoluta','PSI']
corr = df[num_cols].corr(numeric_only=True)
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap (numeric features)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlation_heatmap.png', dpi=160)
plt.close()
corr


,discounted_price_clean,actual_price_clean,discount_pct_clean,rating_clean,rating_count_clean,economia_absoluta,PSI
discounted_price_clean,1.000000,0.962263,-0.236517,0.126365,-0.024055,0.764569,-0.039075
actual_price_clean,0.962263,1.000000,-0.111786,0.127064,-0.034474,0.911110,0.027236
discount_pct_clean,-0.236517,-0.111786,1.000000,-0.161799,0.003856,0.093462,0.482136
rating_clean,0.126365,0.127064,-0.161799,1.000000,0.097550,0.109567,0.502370
rating_count_clean,-0.024055,-0.034474,0.003856,0.097550,1.000000,-0.045218,0.422536
economia_absoluta,0.764569,0.911110,0.093462,0.109567,-0.045218,1.000000,0.123695
PSI,-0.039075,0.027236,0.482136,0.502370,0.422536,0.123695,1.000000


In [6]:
top_by_category = (
    df.sort_values(['main_category','PSI'], ascending=[True, False])
    .groupby('main_category', as_index=False)
    .head(5)
)
top_by_category.to_csv(REPORTS_DIR / 'top_products_by_category_psi.csv', index=False)
top_by_category[['main_category','product_name','PSI','rating_clean','rating_count_clean','discount_pct_clean']].head(15)


,main_category,product_name,PSI,rating_clean,rating_count_clean,discount_pct_clean
1135,Car&Motorbike,Reffair AX30 [MAX] Portable Air Purifier for C...,52.636721,3.8,1118.0,42.0
2,Computers&Accessories,AmazonBasics USB 2.0 Cable - A-Male to B-Male ...,82.887274,4.5,107687.0,70.0
3,Computers&Accessories,AmazonBasics USB 2.0 - A-Male to A-Female Exte...,82.617210,4.5,74976.0,73.0
10,Computers&Accessories,AmazonBasics USB 2.0 Extension Cable for Perso...,79.957675,4.5,74977.0,63.0
14,Computers&Accessories,SanDisk Cruzer Blade 32GB USB Flash Drive,79.017875,4.3,253105.0,56.0
16,Computers&Accessories,SanDisk Ultra Dual 64 GB USB 3.0 OTG Pen Drive...,78.955910,4.3,189104.0,59.0
0,Electronics,"Amazon Basics High-Speed HDMI Cable, 6 Feet (2...",87.744681,4.4,426973.0,78.0
1,Electronics,AmazonBasics Flexible Premium HDMI Cable (Blac...,85.351064,4.4,426973.0,69.0
4,Electronics,ELV Aluminum Adjustable Mobile Phone Foldable ...,82.206872,4.5,28978.0,82.0
5,Electronics,Elv Aluminium Adjustable Mobile Phone Foldable...,81.409000,4.5,28978.0,79.0
